In [ ]:
pip install tavily-python

In [ ]:
from langchain.tools import tool 
import requests
from tavily import TavilyClient
import os
from dotenv import load_dotenv
load_dotenv()

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))


def get_tavily_summary(Query: str) -> str:
    return tavily.search(query=query,max_results=5)

query="How to make a cake?"
print(get_tavily_summary(query))

In [ ]:
from langchain.tools import tool 
import requests
from bs4 import BeautifulSoup
from tavily import TavilyClient
import os 
from dotenv import load_dotenv
from rich import print
load_dotenv()

tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))


@tool
def web_search(query : str) -> str:
    """Search the web for recent and reliable information on a topic . Returns Titles , URLs and snippets."""
    results = tavily.search(query=query,max_results=5)

    out = []

    for r in results['results']:
        out.append(
            f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n"
        )
    
    return "\n----\n".join(out)

query = "Recent advancements in AI research"
print(web_search.run(query))

In [ ]:
response = tavily_client.extract("https://en.wikipedia.org/wiki/Lionel_Messi")

print(response)

In [ ]:
response = tavily_client.extract("https://www.nationalgeographic.com/animals/mammals/facts/domestic-cat")

print(response)

In [ ]:
import re
import textwrap

# Extract and clean the raw content
raw_content = response['results'][0]['raw_content']

# Clean the content
def clean_tavily_content(text):
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    
    # Remove asterisks and other markdown symbols
    text = re.sub(r'\*+', '', text)
    
    # Remove navigation elements and bullets
    text = re.sub(r'^\* ', '', text, flags=re.MULTILINE)
    
    # Clean up definition list formatting (: patterns)
    text = re.sub(r'^:\s+', '', text, flags=re.MULTILINE)
    
    # Remove extra whitespace and newlines
    text = re.sub(r'\n\s*\n', '\n\n', text)  # Multiple newlines to double
    text = re.sub(r'[ \t]+', ' ', text)      # Multiple spaces to single
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?;:()\-"\']', ' ', text)
    
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

# Apply cleaning
clean_content = clean_tavily_content(raw_content)

# Format for better readability
formatted_content = textwrap.fill(clean_content, width=80)

print("CLEANED CONTENT:")
print("=" * 50)
print(formatted_content)

In [ ]:
uv add beautifulsoup4 requests


In [ ]:
html_doc = """
<html><head><title>The Dormouse's story</title></head>
<body>
<p class="title"><b>The Dormouse's story</b></p>

<p class="story">Once upon a time there were three little sisters; and their names were
<a href="http://example.com/elsie" class="sister" id="link1">Elsie</a>,
<a href="http://example.com/lacie" class="sister" id="link2">Lacie</a> and
<a href="http://example.com/tillie" class="sister" id="link3">Tillie</a>;
and they lived at the bottom of a well.</p>

<p class="story">...</p>
"""

soup = BeautifulSoup(html_doc, 'html.parser')

In [ ]:
soup

In [ ]:
prices = soup.find("p")
prices = soup.find_next("p")
prices

In [ ]:
import requests
url = "https://www.nationalgeographic.com/animals/mammals/facts/domestic-cat"
response = requests.get(url)
response.content[:300]

In [ ]:
soup = BeautifulSoup(response.content, 'html.parser')


In [ ]:
for element in soup(['script', 'style', 'nav', 'footer', 'header']):
    element.decompose()


In [ ]:
# Extract main content (adjust selectors as needed)
main_content = soup.find('p')
main_content

In [ ]:
main_content = soup.find_next('main')
main_content

In [ ]:
# Convert to clean text
h = html2text.HTML2Text()
h.ignore_links = True
h.ignore_images = True
clean_text = h.handle(str(main_content))


In [ ]:
extracted=tavily_client.extract(
    url,
    extract_depth="advanced"
)

In [ ]:
extracted

In [ ]:
extracted.keys()

In [ ]:
print(extracted.get('request_id'))

In [ ]:
results=extracted.get("results")
print(results)

In [ ]:
raw_contents = [result['raw_content'] for result in results]


In [ ]:
raw_contents

In [61]:
text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', raw_content)
text = re.sub(r'\*+', '', text)
text = re.sub(r'\n\s*\n', '\n\n', text)
text = re.sub(r'[ \t]+', ' ', text)
text

' Animals\n Photo Ark\n\n# Domestic cat\n\nCommon Name:\n: Domestic Cat\n\nScientific Name:\n: Felis catus\n\nType:\n: Mammals\n\nDiet:\n: Carnivore\n\nSize:\n: 28 inches\n\nWeight:\n: 5 to 20 pounds\n\nSize relative to a 6-ft man:\n\nIUCN Red List Status:\n: Not evaluated\n\nLC\n\nNT\n\nVU\n\nEN\n\nCR\n\nEW\n\nEX\n\nLeast Concern Extinct\n\nCurrent Population Trend:\n: Unknown\n\n## Where do cats come from?\n\nFrom ancient Egyptians to today’s internet users, people have always loved their cats.\n\nIn the U.S. alone, cats reign over about 45.3 million households. There are at least 45 domestic breeds, which differ widely in features such as coat color, tail length, hair texture, and temperament, according to the Cat Fancier’s Association.\n\nThe Maine Coon is the largest, with males reaching an average of 3.5 feet long. The smallest breed is the Singapura, native to Singapore, with adult females weighing as little as four pounds. One of the most unusual-looking cats is the Sphynx, a m

In [64]:
@tool
def scrape_url(url: str) -> str:
    try:
        resp = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)
    except Exception as e:
        return f"Could not scrape URL: {str(e)}"


v =scrape_url("https://cats.com/")

print(v)

Cats.com - In-depth cat product reviews, behavior guides, and vet-written education on symptoms, diseases, and 
more. Cats.com Search Cats.com — In-depth cat product reviews, behavior guides, and vet-written education on 
symptoms, diseases, and more. TOP Articles Cat Breeds 10 Biggest Maine Coon Cats That Are Larger Than Life Maine 
Coon cats and kittens are all over social media and it’s no surprise why. These gentle giants are gorgeous and 
impressively larger than life. Maine Coons (the official state... by                                               
Jackie Brown TOP Articles Cat Health What If You Can’t Afford Veterinary Care For Your Cat? You Have Options 
Throughout your cat’s life, they may need surgery, hospitalization, specialized procedures, or costly medications. 
And that’s on top of their regular checkups. Veterinary medicine has made incredible advancements in recent... by  
Jackie Brown TOP Articles Behavioral Problems in Cats 8 Tips to Stop Your Kitten From Biting Kittens are charming 
creatures with fluffy coats, big eyes, and soft meows. However, their sharp teeth and claws can cause injury. 
Kitten biting is a natural behavior during teething, playtime,... by                                               
Melina Grin Recent Articles Cat Basics What Does Your Cat Do All Day? by  Kate Barrington Litter AIPERRO XL 
Stainless Steel Litter Box Review: Is It Worth Buying? by  Amy Brown-Towry Cat Basics 7 Ways to Keep Your Cat Safe 
& Comfortable on the 4th of July by  Kellie B. Gormly Cat Stories News They Saved a Kitten During a Hurricane by  
Katelynn Sobus News This ‘Diva’ Cat Throws Shade Playing Cards Against Humanity And It’s Hysterical by  Cats.com 
Editorial Team Don’t miss 1. The 12 Healthy Best Dry Cat Foods of 2026 2. 12 Best Cat Brushes & Deshedding Tools 3.
From Sick Rescue Cat to Spoiled Princess 2,150+ Cat Guides Published 20+ veterinary advisors 10M+ cat owners helped
every year Newsletter All you and your cat need in 5 minutes Join over 1 million cat lovers and learn how to win 
your cat’s affection score exclusive deals, get tips to better understand your cat, and stay updated with the 
latest cat news. *by subscribing you agree to receive emails from cats.com Where Cat Lovers and Experts Meet. At 
Cats.com community we’re dedicated to helping every cat live a happier, healthier life. From nutrition and health 
to behavior and bonding, we bring together expert advice, trusted resources, and a passionate community of cat 
lovers. Free Cat Health & Care Courses Step-by-step guides on cat care, behavior, and wellbeing. Community Forums 
Connect with fellow cat lovers, share stories, and ask questions. Expert Advices Video Library Insights from vets 
and behaviorists you can trust. Helpful Tools Resources designed to support you and your cat every day. Sign Up for
Free Log In Popular in the Community Cat Behavior Are My Cats Playing or Fighting? Are you trying to figure out if 
your cats are playing or... by  Melina Grin Cat Studies Research Suggests Cats Possess Healing Powers Our cats are 
very special. They shape our lives and make us... by  Cats.com Editorial Team Cat Safety Advice Top 10 Things Your 
Vet Wishes You Knew It would be so much easier if our cats came with instruction... by  Dr. Sarah Wooten, DVM, CVJ 
Cat Behavior Can Cats See Ghosts and Spirits? Have you ever seen a scary movie where the dog or cat... by  Mallory 
Crusta Cat Behavior Are Cats Smarter Than Dogs? Scientists May Finally Have the Answer As the two most popular pets
in the world, it’s impossible not... by  Amber King Cat Medication Benadryl for Cats: Dosage, Safety & Side Effects
Benadryl, which is a brand name for the drug diphenhydramine hydrochloride, is... by  Jackie Brown Cat Breeds The 
10 Friendliest & Nicest Cat Breeds In the World Although cats have earned a reputation for being independent and 
aloof, cat... by  Jackie Brown Cat Nutrition & Diet Wet vs. Dry Cat Food: What’s Better for Cats? Cat owners ofte